In [2]:
%load_ext autoreload
%autoreload 2
import sys, pathlib, yaml
import pandas as pd

sys.path.append(str(pathlib.Path.cwd().parent))
from src.clean import clean
from src.normalize import normalize, report_unknown_brands


In [3]:
with open("../config/config.yaml") as f:
    cfg = yaml.safe_load(f)
with open("../config/brands.yaml") as f:
    brand_config = yaml.safe_load(f)

raw = pd.read_csv("../" + cfg["paths"]["raw"])

In [4]:
cleaned, quarantined = clean(
    raw,
    placeholder_price_threshold=cfg["cleaning"]["placeholder_price_threshold_toman"],
    extreme_high_threshold=cfg["cleaning"]["extreme_high_price_threshold_toman"],
    mileage_high_threshold=cfg["cleaning"]["mileage_high_threshold_km"],
)
normalized = normalize(cleaned, brand_config)

print("Unmatched brands (candidates to add to config/brands.yaml):")
print(report_unknown_brands(cleaned, brand_config))

for key in ["clean", "quarantine", "normalized"]:
    out_path = pathlib.Path("../" + cfg["paths"][key])
    out_path.parent.mkdir(parents=True, exist_ok=True)

cleaned.to_csv("../" + cfg["paths"]["clean"], index=False, encoding="utf-8-sig")
quarantined.to_csv("../" + cfg["paths"]["quarantine"], index=False, encoding="utf-8-sig")
normalized.to_csv("../" + cfg["paths"]["normalized"], index=False, encoding="utf-8-sig")

Cleaning summary
  raw rows                          : 1800
  after dedup                       : 1798  (-2)
  after rental/non-passenger drop    : 1716  (-82)
  after essential-field dropna       : 1710  (-6)
  after placeholder-price drop (<100,000,000): 1626  (-84)
  extreme-high prices corrected (/1000, comparable-validated): 2
  extreme-high prices still flagged (unresolved)   : 0
  mileage outlier flags              : 6
Unmatched brands (candidates to add to config/brands.yaml):
brand_model
پژو 206 تیپ ۲                    82
پژو 207i پانوراما دنده ای        55
پراید 131 SE                     55
پراید صندوق دار بنزینی           43
پژو 405 GLX بنزینی               34
                                 ..
پراید وانت 151 SL                 1
چانگان CS 35 پلاس تیپ 2           1
فونیکس FX پرمیوم دو دیفرانسیل     1
مرسدس بنز C Class C200L           1
مکث موتور کلوت دنده ای            1
Name: count, Length: 397, dtype: int64
